# Gaussian appendix regeneration — Tables 6, 7, 9, 10, 13 on one device

Pre-registered configuration: `taskc/DECISIONS.md` §16 (nothing here may change it). Everything — both DDPM trainings, the 2-D model, the estimated-SMT dual and h_ψ, every sampling configuration — runs on **this** device, so every row of every table comes from the same machine. Results are written to Drive after every stage.

No end-of-process mean adjustment anywhere; de-standardization `Y = s0·z + mP` once. The published `src/` and `src_v2/` code is used unchanged (1-dim `ScoreMLP`, cosine schedule, `rn_noise_shift_std`, reverse-step arithmetic via `taskc.sampler.reverse_ancestral`).

Batched sampling: all eight exact-shift configurations (λ = −1 … 1.5, which include the no-shift row λ=0 and the exact-SMT row λ=1) share **one** reverse pass with a per-sample λ; retrained-Q and estimated-SMT (autograd hook) are two further passes. Each configuration draws one i.i.d. pool of 2.52 M returns that serves both the H=21 tables (first 10⁵ × 21) and the maturity table (first 2×10⁴ × 126).

Runtime on an A100: ≈ 10–15 min end to end.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git log --oneline -1

In [ ]:
import os, sys, json, math, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, torch
import matplotlib.pyplot as plt
from dataclasses import asdict, replace
import taskc
from taskc.gauss_appendix import (GaussCfg, train_1d, draw_z, draw_z_lambda_batched, evaluate_config, pool_size,
                                  dual_1d, logL_1d, tables_markdown, paths_from_returns)
from taskc.sampler import last_unclipped_step
from taskc.hnet import HNet, train_hnet, grad_log_h, save_hnet
from taskc.ptheta import Schedule
from src.schedules import make_alpha_schedule
from src_v2.diffusion.score_mlp import ScoreMLP as ScoreMLPND
from scipy.stats import wasserstein_distance

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Output location (Drive) and the pre-registered configuration

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/ddpm_option_pricing/artifacts_gauss_colab"
else:
    OUT = "artifacts_gauss_colab"        # local dry run
os.makedirs(OUT, exist_ok=True)

QUICK = False                            # True -> tiny sizes for a plumbing check (not for the paper)
cfg = GaussCfg(chunk=1_000_000 if DEVICE == "cuda" else 200_000)
if QUICK:
    cfg = replace(cfg, N_train=20_000, epochs=40, n_paths=5_000, n_paths_mat=2_000, n_lambda=2_000, chunk=50_000)
print(json.dumps(asdict(cfg), indent=1))
print(f"s0={cfg.s0:.6f}  mP={cfg.mP:.4e}  mQ={cfg.mQ:.4e}  d_std={cfg.d_std:+.5f}  pool per configuration = {pool_size(cfg):,} returns")

sched = make_alpha_schedule(T=cfg.T, device=torch.device(DEVICE), s=cfg.cosine_s)
t_start = last_unclipped_step(sched[0])
S = Schedule(sched[0], sched[1], sched[2], t_start)
rng = np.random.default_rng(cfg.seed_draw)
res = dict(config=asdict(cfg), derived=dict(s0=cfg.s0, mP=cfg.mP, mQ=cfg.mQ, d_std=cfg.d_std, t_start=t_start),
           device=DEVICE, device_name=(torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE),
           seeds=dict(data_P=cfg.seed_data_P, data_Q=cfg.seed_data_Q, init_P=cfg.seed_init_P, init_Q=cfg.seed_init_Q, draw_base=cfg.seed_draw),
           timings={}, table6={}, table7={}, table9={}, table10={}, table13={})
def save():
    json.dump(res, open(os.path.join(OUT, "gauss_appendix.json"), "w"), indent=1)
    open(os.path.join(OUT, "tables.md"), "w").write(tables_markdown(res, cfg))
save(); print("writing to", OUT)

## Train P_θ, retrained-Q, and the 2-D model (same recipe, same device)

In [ ]:
yP = np.random.default_rng(cfg.seed_data_P).normal(cfg.mP, cfg.s0, cfg.N_train)
yQ = np.random.default_rng(cfg.seed_data_Q).normal(cfg.mQ, cfg.s0, cfg.N_train)
zP, zQ = (yP - cfg.mP) / cfg.s0, (yQ - cfg.mP) / cfg.s0          # one (source) standardizer for both
t0 = time.time(); modelP = train_1d(zP, sched, cfg, DEVICE, cfg.seed_init_P); res["timings"]["train_P_s"] = time.time() - t0
t0 = time.time(); modelQ = train_1d(zQ, sched, cfg, DEVICE, cfg.seed_init_Q); res["timings"]["train_Q_s"] = time.time() - t0
mP2, mQ2, s2 = np.array([0.0, 0.0]), np.array([2.0, 1.0]), np.array([0.5, 0.5])
z2 = (np.random.default_rng(7).normal(mP2, s2, (cfg.N_train, 2)) - mP2) / s2
torch.manual_seed(0); model2 = ScoreMLPND(data_dim=2, hidden_dim=cfg.hidden, time_emb_dim=cfg.time_emb)
t0 = time.time(); model2 = train_1d(z2, sched, cfg, DEVICE, 0, model=model2); res["timings"]["train_2d_s"] = time.time() - t0
torch.save(dict(P=modelP.state_dict(), Q=modelQ.state_dict(), D2=model2.state_dict(), cfg=asdict(cfg), device=DEVICE), os.path.join(OUT, "models.pt"))
print({k: f"{v/60:.1f} min" for k, v in res["timings"].items()}); save()

## Estimated SMT: dual on 10⁶ P_θ draws → L* → h_ψ (non-circular; no analytic target)

In [ ]:
nA, nB = (1_000_000, 200_000) if not QUICK else (50_000, 20_000)
t0 = time.time()
zA, _ = draw_z(modelP, sched, nA, 1, cfg.seed_draw + 1, DEVICE, cfg, t_start=t_start)
dual = dual_1d(cfg.s0 * zA[:, 0] + cfg.mP, cfg)
zB, _ = draw_z(modelP, sched, nB, 1, cfg.seed_draw + 2, DEVICE, cfg, t_start=t_start)
LB = np.exp(logL_1d(cfg.s0 * zB[:, 0] + cfg.mP, dual, cfg))
hnet = HNet(1, 256, 32, 0.05)
hlog = train_hnet(hnet, zB.astype(np.float32), LB, S, device=DEVICE, epochs=100 if not QUICK else 5, verbose=False)
save_hnet(os.path.join(OUT, "hpsi_gauss.pt"), hnet, dict(dual=dual, E_L_B=float(LB.mean()), epochs=100 if not QUICK else 5, device=DEVICE))
res["estimated_smt"] = dict(dual=dual, E_L_B=float(LB.mean()), L_min=float(LB.min()), L_max=float(LB.max()), mse=hlog.epoch_loss, floor_max=max(hlog.floor_frac))
res["timings"]["estimated_smt_setup_s"] = time.time() - t0; save()
print(f"dual: beta_raw {dual['beta_raw']:.4f}  ESS {dual['ess']*100:.2f}%  fitted {dual['fitted']:.1e} | h_psi: E_B[L*] {LB.mean():.4f}  L* in [{LB.min():.3f}, {LB.max():.3f}]  MSE {hlog.epoch_loss[-1]:.2e}  floor {max(hlog.floor_frac):.1e}  ({(time.time()-t0)/60:.1f} min)")
abar_dev = sched[2]
def est_hook(y, t01, t):
    return -torch.sqrt(1 - abar_dev[t]) * grad_log_h(hnet, y, t01, abar_dev[t].expand(y.shape[0]))

## Sampling — three shared reverse passes

Pass 1: all λ rows at once (incl. no-shift λ=0 and exact-SMT λ=1). Pass 2: retrained Q. Pass 3: estimated SMT (autograd of log h_ψ per step).

In [ ]:
n_pool = pool_size(cfg)
n_lam = max(cfg.n_lambda * cfg.H, 0)
sizes = {lam: (n_pool if lam in (0.0, 1.0) else n_lam) for lam in cfg.lambdas}
t0 = time.time()
zl, rej_l = draw_z_lambda_batched(modelP, sched, list(cfg.lambdas), n_pool, cfg.seed_draw + 10, DEVICE, cfg, t_start=t_start)
res["timings"]["pass_lambda_s"] = time.time() - t0; print(f"pass 1 (8 lambda rows x {n_pool:,}): {(time.time()-t0)/60:.1f} min, rejections {rej_l}")
t0 = time.time(); zQ_, rej_q = draw_z(modelQ, sched, n_pool, 1, cfg.seed_draw + 11, DEVICE, cfg, t_start=t_start); zQ_ = zQ_[:, 0]
res["timings"]["pass_retrained_s"] = time.time() - t0; print(f"pass 2 (retrained Q): {(time.time()-t0)/60:.1f} min, rejections {rej_q}")
t0 = time.time(); zE, rej_e = draw_z(modelP, sched, n_pool, 1, cfg.seed_draw + 12, DEVICE, cfg, eps_correction=est_hook, t_start=t_start); zE = zE[:, 0]
res["timings"]["pass_estimated_s"] = time.time() - t0; print(f"pass 3 (estimated SMT): {(time.time()-t0)/60:.1f} min, rejections {rej_e}")
np.savez_compressed(os.path.join(OUT, "pools.npz"), **{f"lam_{lam}": zl[lam].astype(np.float32) for lam in cfg.lambdas}, retrainedQ=zQ_.astype(np.float32), estimated=zE.astype(np.float32))
save()

## Tables 6, 7, 9, 10

In [ ]:
pools = {"No shift": zl[0.0], "Exact SMT": zl[1.0], "Estimated SMT": zE, "Retrained Q": zQ_}
for name, z in pools.items():
    e = evaluate_config(z, cfg, rng)
    res["table6"][name] = e["h21"]; res["table10"][name] = e["h21"]["terminal"]; res["table9"][name] = {str(H): v for H, v in e["maturity"].items()}
for lam in cfg.lambdas:
    z = zl[lam]
    if lam in (0.0, 1.0):
        r = res["table6"]["No shift" if lam == 0.0 else "Exact SMT"]; res["table7"][str(lam)] = dict(mart_dev=r["mart_dev"], rmse=r["rmse"], W1=r["W1"], n=cfg.n_paths, source="table 6 row")
    else:
        e = evaluate_config(z[:cfg.n_lambda * cfg.H if cfg.n_lambda * cfg.H <= len(z) else len(z)], cfg, rng, n_h21=cfg.n_lambda, n_mat=1)["h21"]
        res["table7"][str(lam)] = dict(mart_dev=e["mart_dev"], rmse=e["rmse"], W1=e["W1"], n=cfg.n_lambda, source="run")
save()
print(tables_markdown(res, cfg))

## Table 13 — 2-D transport, no adjustment

In [ ]:
d2 = torch.tensor((mQ2 - mP2) / s2, dtype=torch.float32, device=DEVICE); abar = sched[2]
hook2 = lambda y, t01, t: -(torch.sqrt(abar[t] * (1 - abar[t])) * d2).view(1, 2).expand_as(y)
n2 = cfg.n_paths
t0 = time.time()
z_no = draw_z(model2, sched, n2, 2, 501, DEVICE, cfg, t_start=t_start)[0]; z_smt = draw_z(model2, sched, n2, 2, 502, DEVICE, cfg, hook2, t_start)[0]
Qref = np.random.default_rng(9).normal(mQ2, s2, (n2, 2))
for k, v in (("No shift", z_no), ("SMT", z_smt)):
    x = v * s2 + mP2
    res["table13"][k] = dict(mean=x.mean(0).tolist(), std=x.std(0).tolist(), W1_x1=float(wasserstein_distance(x[:, 0], Qref[:, 0])), W1_x2=float(wasserstein_distance(x[:, 1], Qref[:, 1])))
res["table13"]["Q_ref"] = dict(mean=Qref.mean(0).tolist(), std=Qref.std(0).tolist()); res["table13"]["target"] = dict(mean=mQ2.tolist(), std=s2.tolist())
res["timings"]["table13_s"] = time.time() - t0; res["timings"]["total_s"] = sum(res["timings"].values()); save()
print(tables_markdown(res, cfg).split("**Table 13")[1]); print("total", f"{res['timings']['total_s']/60:.1f} min")

## Diagnostics (inline)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for name in ("No shift", "Exact SMT", "Estimated SMT", "Retrained Q"):
    ax[0].plot(res["table6"][name]["mart_profile"], label=name)
ax[0].axhline(0, c="k", lw=.8); ax[0].set_title("E[e^{-rt}S_t] - S0 vs step"); ax[0].legend(fontsize=7)
lams = sorted(float(k) for k in res["table7"]); ax[1].plot(lams, [res["table7"][str(l)]["mart_dev"] for l in lams], marker="o", label="mart. dev."); ax[1].plot(lams, [res["table7"][str(l)]["rmse"] for l in lams], marker="x", label="RMSE"); ax[1].set_xlabel("lambda"); ax[1].legend(); ax[1].set_title("lambda sweep")
for name in ("No shift", "Exact SMT", "Estimated SMT", "Retrained Q"):
    ax[2].plot(cfg.maturities, [res["table9"][name][str(H)]["atm"] - res["table9"][name][str(H)]["bs"] for H in cfg.maturities], marker="o", label=name)
ax[2].axhline(0, c="k", lw=.8); ax[2].set_title("ATM price - BS vs H"); ax[2].legend(fontsize=7)
ST = {n: paths_from_returns((cfg.s0 * pools[n][:cfg.n_paths*cfg.H] + cfg.mP).reshape(cfg.n_paths, cfg.H), cfg.S0)[:, -1] for n in pools}
bins = np.linspace(80, 125, 90)
for n in ("No shift", "Exact SMT", "Retrained Q"): ax[3].hist(ST[n], bins=bins, density=True, histtype="step", label=n)
ax[3].set_title("S_21"); ax[3].legend(fontsize=7); plt.tight_layout(); plt.show()
print("no-shift mean-return bias vs mP:", f"{res['table6']['No shift']['bias_sd_vs_mP']:+.4f} sd", "| exact-SMT bias vs mQ:", f"{res['table6']['Exact SMT']['bias_sd_vs_mQ']:+.4f} sd", "| shift d =", f"{cfg.d_std:+.4f} sd")

Done. `gauss_appendix.json`, `tables.md`, `models.pt`, `hpsi_gauss.pt`, `pools.npz` are in the Drive folder. Numbers go to `DECISIONS.md` §16 and the manuscript is touched only after that.